# Cheng NN: 45-year time-varying GEV estimator

This notebook trains a simulation-based estimator for annual maxima with linear time effects in $\mu(t)$ and $\log\sigma(t)$ and a time-constant $\xi$. It is independent of the existing stationary 11-quantile NN.

All execution switches are `False` by default. Creating this notebook does not generate data or train a model.

## Model definition

For $t_i^c=(\operatorname{year}_i-2002)/10$,

$$\mu(t)=\mu_0+\beta_\mu t^c,$$

$$\log\sigma(t)=\eta_0+\beta_\sigma t^c,\qquad \xi(t)=\xi_0.$$

The input has shape $45\times2$, with columns $(t_i^c,Z_i)$. The output is

$$\boldsymbol\theta=(\mu_0,\beta_\mu,\eta_0,\beta_\sigma,\xi_0)\in\mathbb{R}^5.$$

In [ ]:
from pathlib import Path
import copy
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT if (CURRENT / 'src').exists() else CURRENT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cheng_nn_simulation import (
    COEFFICIENT_NAMES, ParameterRanges, SimulationConfig,
    generate_all_splits,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'simulated' / 'cheng_nn'
MODEL_PATH = PROJECT_ROOT / 'models' / 'cheng_nn_time_varying.pt'

RUN_SIMULATION = False
RUN_TRAINING = False
RUN_EVALUATION = False

N_TRAIN = 300_000
N_VALIDATION = 40_000
N_TEST = 40_000
BATCH_SIZE = 128
MAX_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 20
NUM_WORKERS = 0  # safest default for Windows notebooks
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DEVICE =', DEVICE)
print('DATA_DIR =', DATA_DIR)

## 1. Generate independent simulation splits

Each simulated item is one virtual GRID with 45 annual maxima. Training, validation, and test use independent random seeds. Parameter ranges below are preliminary design ranges and must be checked against the intended global or Taiwan application before final training.

In [ ]:
SIMULATION_CONFIG = SimulationConfig(
    start_year=1980,
    n_years=45,
    time_scale_years=10.0,
    chunk_size=4096,
    seed=20260923,
)
PARAMETER_RANGES = ParameterRanges(
    mu0=(20.0, 45.0),
    beta_mu=(-1.0, 1.0),
    eta0=(float(np.log(0.5)), float(np.log(5.0))),
    beta_sigma=(-0.20, 0.20),
    xi0=(-0.40, 0.40),
)

if RUN_SIMULATION:
    generated_directories = generate_all_splits(
        DATA_DIR,
        n_train=N_TRAIN,
        n_validation=N_VALIDATION,
        n_test=N_TEST,
        config=SIMULATION_CONFIG,
        ranges=PARAMETER_RANGES,
    )
    display(generated_directories)
else:
    print('Simulation skipped. Set RUN_SIMULATION=True when ready.')

### Saved arrays

- `inputs.npy`: $(N,45,2)$, containing centered time and standardized maxima.
- `targets_standardized.npy`: $(N,5)$, the labels used for NN training.
- `coefficients_original.npy`: $(N,5)$, the known original-scale truth.
- `sample_location_scale.npy`: sample median and IQR for inverse transformation.

In [ ]:
def load_split(split_name):
    split_dir = DATA_DIR / split_name
    required = {
        'inputs': split_dir / 'inputs.npy',
        'targets': split_dir / 'targets_standardized.npy',
        'coefficients': split_dir / 'coefficients_original.npy',
        'location_scale': split_dir / 'sample_location_scale.npy',
        'metadata': split_dir / 'metadata.json',
    }
    missing = [str(path) for path in required.values() if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing simulation outputs:\n' + '\n'.join(missing))
    return {
        'inputs': np.load(required['inputs'], mmap_mode='r'),
        'targets': np.load(required['targets'], mmap_mode='r'),
        'coefficients': np.load(required['coefficients'], mmap_mode='r'),
        'location_scale': np.load(required['location_scale'], mmap_mode='r'),
        'metadata': json.loads(required['metadata'].read_text(encoding='utf-8')),
    }

datasets_available = all((DATA_DIR / name).exists() for name in ('train', 'validation', 'test'))
if datasets_available:
    train_data = load_split('train')
    validation_data = load_split('validation')
    test_data = load_split('test')
    print('Train input shape:', train_data['inputs'].shape)
    print('Train target shape:', train_data['targets'].shape)
else:
    train_data = validation_data = test_data = None
    print('Simulation datasets do not exist yet; nothing was loaded.')

## 2. Five-layer neural estimator

The baseline retains the five hidden ReLU layers used in the stationary NN paper, but changes the input from 11 quantiles to the ordered $45\times2$ sequence and changes the output from 3 to 5 coefficients.

In [ ]:
class ChengGEVNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(45 * 2, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 5),
        )

    def forward(self, x):
        if x.ndim != 3 or x.shape[1:] != (45, 2):
            raise ValueError(f'Expected (batch, 45, 2), received {tuple(x.shape)}')
        return self.network(x)

model_summary = ChengGEVNet()
print(model_summary)
print('Trainable parameters =', sum(p.numel() for p in model_summary.parameters() if p.requires_grad))

In [ ]:
class TargetScaler:
    def __init__(self, mean, std):
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)
        if self.mean.shape != (5,) or self.std.shape != (5,):
            raise ValueError('Target mean and std must each have shape (5,).')
        if np.any(self.std <= 1e-8):
            raise ValueError('Every target must have nonzero training variation.')

    @classmethod
    def fit(cls, targets):
        return cls(np.mean(targets, axis=0), np.std(targets, axis=0))

    def transform(self, values):
        return (np.asarray(values) - self.mean) / self.std

    def inverse_transform(self, values):
        return np.asarray(values) * self.std + self.mean

    def state_dict(self):
        return {'mean': self.mean, 'std': self.std}


class ChengSimulationDataset(Dataset):
    def __init__(self, inputs, targets, target_scaler):
        if len(inputs) != len(targets):
            raise ValueError('Input and target sample counts differ.')
        self.inputs = inputs
        self.targets = targets
        self.target_scaler = target_scaler

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        x = np.array(self.inputs[index], dtype=np.float32, copy=True)
        y = self.target_scaler.transform(self.targets[index]).astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)

## 3. Training

The five target coefficients are standardized using training-set means and standard deviations before computing MSE. This prevents the location coefficient from dominating the smaller trend and shape coefficients.

In [ ]:
def mean_batch_loss(model, loader, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_count = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs)
            loss = torch.mean((predictions - targets) ** 2)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            total_loss += float(loss.detach()) * len(inputs)
            total_count += len(inputs)
    return total_loss / total_count


def train_cheng_model(train_loader, validation_loader, max_epochs, patience, device):
    model = ChengGEVNet().to(device)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=6, min_lr=1e-6
    )
    best_state = None
    best_validation = float('inf')
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        started = time.perf_counter()
        train_loss = mean_batch_loss(model, train_loader, device, optimizer)
        validation_loss = mean_batch_loss(model, validation_loader, device)
        scheduler.step(validation_loss)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'validation_loss': validation_loss,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'seconds': time.perf_counter() - started,
        })
        print(
            f'Epoch {epoch:03d}: train={train_loss:.6f}, '
            f'validation={validation_loss:.6f}'
        )

        if validation_loss < best_validation - 1e-7:
            best_validation = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print('Early stopping.')
                break

    if best_state is None:
        raise RuntimeError('Training did not produce a valid checkpoint.')
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
if RUN_TRAINING:
    if not datasets_available:
        raise RuntimeError('Generate the simulation datasets before training.')

    target_scaler = TargetScaler.fit(train_data['targets'])
    train_dataset = ChengSimulationDataset(
        train_data['inputs'], train_data['targets'], target_scaler
    )
    validation_dataset = ChengSimulationDataset(
        validation_data['inputs'], validation_data['targets'], target_scaler
    )
    loader_options = {
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'pin_memory': DEVICE == 'cuda',
    }
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
    validation_loader = DataLoader(
        validation_dataset, shuffle=False, **loader_options
    )
    model, training_history = train_cheng_model(
        train_loader, validation_loader, MAX_EPOCHS,
        EARLY_STOPPING_PATIENCE, DEVICE,
    )

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state': model.state_dict(),
        'target_scaler': target_scaler.state_dict(),
        'coefficient_names': list(COEFFICIENT_NAMES),
        'input_shape': [45, 2],
        'simulation_metadata': train_data['metadata'],
    }, MODEL_PATH)
    display(training_history.tail())
else:
    model = target_scaler = training_history = None
    print('Training skipped. Set RUN_TRAINING=True when ready.')

## 4. Independent simulation test

Report RMSE and MAE against the known generating coefficients on the untouched test split. Do not use the test split for early stopping or architecture selection.

In [ ]:
def load_cheng_checkpoint(path, device):
    try:
        checkpoint = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=device)
    loaded_model = ChengGEVNet().to(device)
    loaded_model.load_state_dict(checkpoint['model_state'])
    loaded_model.eval()
    scaler = TargetScaler(**checkpoint['target_scaler'])
    return loaded_model, scaler, checkpoint


def predict_standardized_coefficients(model, inputs, target_scaler, device, batch_size=1024):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(inputs), batch_size):
            batch = np.array(inputs[start:start + batch_size], dtype=np.float32, copy=True)
            normalized = model(torch.from_numpy(batch).to(device)).cpu().numpy()
            predictions.append(target_scaler.inverse_transform(normalized))
    return np.concatenate(predictions, axis=0)


def inverse_sample_standardization(coefficients, location_scale):
    coefficients = np.asarray(coefficients, dtype=np.float64)
    median = np.asarray(location_scale[:, 0], dtype=np.float64)
    iqr = np.asarray(location_scale[:, 1], dtype=np.float64)
    original = np.empty_like(coefficients)
    original[:, 0] = coefficients[:, 0] * iqr + median
    original[:, 1] = coefficients[:, 1] * iqr
    original[:, 2] = coefficients[:, 2] + np.log(iqr)
    original[:, 3] = coefficients[:, 3]
    original[:, 4] = coefficients[:, 4]
    return original


if RUN_EVALUATION:
    if not datasets_available:
        raise RuntimeError('The independent test split is missing.')
    evaluation_model, evaluation_scaler, _ = load_cheng_checkpoint(MODEL_PATH, DEVICE)
    predicted_standardized = predict_standardized_coefficients(
        evaluation_model, test_data['inputs'], evaluation_scaler, DEVICE
    )
    predicted_original = inverse_sample_standardization(
        predicted_standardized, test_data['location_scale']
    )
    true_original = np.asarray(test_data['coefficients'], dtype=np.float64)
    errors = predicted_original - true_original
    metrics = pd.DataFrame({
        'coefficient': COEFFICIENT_NAMES,
        'RMSE': np.sqrt(np.mean(errors ** 2, axis=0)),
        'MAE': np.mean(np.abs(errors), axis=0),
        'bias': np.mean(errors, axis=0),
    })
    display(metrics.round(4))
else:
    print('Evaluation skipped. Set RUN_EVALUATION=True after training.')

## 5. Apply the frozen estimator to one real 45-year sequence

This helper does not fit the NN on the real series. It applies a previously trained checkpoint and returns coefficients on the original temperature scale.

In [ ]:
def prepare_real_input(years, annual_maxima):
    years = np.asarray(years, dtype=np.float64)
    values = np.asarray(annual_maxima, dtype=np.float64)
    if years.shape != (45,) or values.shape != (45,):
        raise ValueError('Exactly 45 years and 45 annual maxima are required.')
    if np.any(~np.isfinite(years)) or np.any(~np.isfinite(values)):
        raise ValueError('The real sequence cannot contain missing values.')
    if np.any(np.diff(years) <= 0):
        raise ValueError('Years must be strictly increasing.')

    median = float(np.median(values))
    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = float(q3 - q1)
    if iqr <= 1e-12:
        raise ValueError('The annual maxima have an invalid IQR.')

    centered_time = (years - np.mean(years)) / 10.0
    standardized = (values - median) / iqr
    inputs = np.column_stack((centered_time, standardized)).astype(np.float32)
    return inputs, median, iqr


def estimate_real_sequence(years, annual_maxima, checkpoint_path=MODEL_PATH, device=DEVICE):
    fitted_model, fitted_scaler, _ = load_cheng_checkpoint(checkpoint_path, device)
    inputs, median, iqr = prepare_real_input(years, annual_maxima)
    tensor = torch.from_numpy(inputs[None, :, :]).to(device)
    with torch.no_grad():
        normalized = fitted_model(tensor).cpu().numpy()
    standardized_coefficients = fitted_scaler.inverse_transform(normalized)
    original_coefficients = inverse_sample_standardization(
        standardized_coefficients, np.array([[median, iqr]])
    )[0]
    return dict(zip(COEFFICIENT_NAMES, original_coefficients))

# Example after training:
# estimate = estimate_real_sequence(years_1980_2024, annual_maxima_1980_2024)
# display(estimate)

## Required checks before scientific use

1. Confirm that the real-series summaries fall inside the simulation design.
2. Report coefficient RMSE, MAE, bias, and boundary-stratified errors on the independent simulation test set.
3. Compare this five-layer MLP with a smaller MLP and a time-aware 1D CNN.
4. Use parametric bootstrap after fitting to quantify uncertainty; bootstrap is not a substitute for labelled simulation training.
5. Train separate $M_0$, $M_\mu$, $M_\sigma$, and $M_{\mu\sigma}$ estimators if formal time-structure selection is required.